# Directionality diagnostics

Exploratory Step 3 diagnostics for cell 16. These plots are not pipeline outputs; they decompose the favorable criterion and test wind-threshold sensitivity.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from src.config import load_config
from src.favorable import sector_azimuths, wind_component_along_sector
from src.viz import plotting_config

CELL = 16
FIG_DIR = Path("../docs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

config = load_config("../configs/spb_default.yaml")
favorable = xr.open_zarr("../data/interim/favorable.zarr", consolidated=False)
era5 = xr.open_zarr("../data/interim/era5_spb.zarr", consolidated=False)
stability = xr.open_zarr("../data/interim/stability.zarr", consolidated=False)

az = favorable["sector_azimuth_deg"].values.astype(float)
theta = np.deg2rad(np.r_[az, az[0]])
lat = float(favorable["latitude"].isel(cell=CELL))
lon = float(favorable["longitude"].isel(cell=CELL))


def closed_values(da):
    values = da.values.astype(float)
    return np.r_[values, values[0]]


def setup_polar(ax):
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_thetagrids(np.arange(0, 360, 45), labels=["N", "NE", "E", "SE", "S", "SW", "W", "NW"])
    ax.grid(True, alpha=0.35)

## Favorability decomposition

Outer curve: combined favorable flag. Middle curve: wind-only flag. Inner curve: thermal flag broadcast across sectors.

In [ ]:
combined = favorable["favorable"].isel(cell=CELL).mean("time")
wind = favorable["favorable_wind"].isel(cell=CELL).mean("time")
thermal_scalar = favorable["favorable_thermal"].isel(cell=CELL).mean("time")
thermal = xr.full_like(combined, float(thermal_scalar))

with plt.rc_context(plotting_config()):
    fig, ax = plt.subplots(
        figsize=(5.1, 5.1), subplot_kw={"projection": "polar"}, constrained_layout=True
    )
    setup_polar(ax)
    ax.plot(theta, closed_values(combined), color="#222222", lw=2.0, label="favorable")
    ax.plot(theta, closed_values(wind), color="#1f77b4", lw=1.8, label="favorable_wind")
    ax.plot(theta, closed_values(thermal), color="#d62728", lw=1.6, label="favorable_thermal")
    ax.set_ylim(0, 0.75)
    ax.set_yticks([0.25, 0.50, 0.75])
    ax.set_yticklabels(["0.25", "0.50", "0.75"])
    ax.set_title(f"Criterion decomposition, cell {CELL} ({lat:.2f}N, {lon:.2f}E)")
    ax.legend(loc="lower left", bbox_to_anchor=(-0.02, -0.08), frameon=False)
    fig.savefig(FIG_DIR / "directionality_decomposition_cell16.pdf")
    fig.savefig(FIG_DIR / "directionality_decomposition_cell16.png")
    plt.show()

dict(
    combined_mean=float(combined.mean()),
    wind_mean=float(wind.mean()),
    thermal_flat=float(thermal_scalar),
    combined_peak_az=float(az[int(combined.argmax("sector"))]),
    wind_peak_az=float(az[int(wind.argmax("sector"))]),
)

## Wind-threshold sensitivity

Recompute the wind component from `era5_spb.zarr`; stability is loaded above for parity with Step 3 but intentionally not used in this wind-only diagnostic.

In [ ]:
u = era5["u10"].stack(cell=("latitude", "longitude")).reset_index("cell").isel(cell=CELL)
v = era5["v10"].stack(cell=("latitude", "longitude")).reset_index("cell").isel(cell=CELL)
component = wind_component_along_sector(u, v, sector_azimuths(int(config["sectors"]["count"])))

thresholds = [1.0, 2.0, 3.0]
wind_probs = {threshold: (component >= threshold).mean("time") for threshold in thresholds}

with plt.rc_context(plotting_config()):
    fig, ax = plt.subplots(
        figsize=(5.1, 5.1), subplot_kw={"projection": "polar"}, constrained_layout=True
    )
    setup_polar(ax)
    for threshold, color in zip(thresholds, ["#2ca02c", "#1f77b4", "#9467bd"]):
        ax.plot(
            theta,
            closed_values(wind_probs[threshold]),
            color=color,
            lw=1.8,
            label=f"{threshold:.0f} m/s",
        )
    ax.set_ylim(0, 0.75)
    ax.set_yticks([0.25, 0.50, 0.75])
    ax.set_yticklabels(["0.25", "0.50", "0.75"])
    ax.set_title(f"Wind-threshold sensitivity, cell {CELL} ({lat:.2f}N, {lon:.2f}E)")
    ax.legend(loc="lower left", bbox_to_anchor=(-0.02, -0.08), frameon=False, title="Threshold")
    fig.savefig(FIG_DIR / "wind_threshold_sensitivity_cell16.pdf")
    fig.savefig(FIG_DIR / "wind_threshold_sensitivity_cell16.png")
    plt.show()

{
    threshold: {
        "mean": float(prob.mean()),
        "min": float(prob.min()),
        "max": float(prob.max()),
        "peak_az": float(az[int(prob.argmax("sector"))]),
    }
    for threshold, prob in wind_probs.items()
}

In [ ]:
# 1. Get the polar plot setup from the notebook code
# 2. Print the raw favorable_wind values for cell 16, threshold 2 m/s,
#    by sector azimuth — directly from the data, not from the plot.
import xarray as xr

fav = xr.open_zarr("data/interim/favorable.zarr")
wind_by_sector = fav.favorable_wind.isel(cell=16).mean("time")
for sec_idx, p in zip(range(18), wind_by_sector.values):
    az = sec_idx * 20
    print(f"Sector {sec_idx:2d} (az={az:3d}°): p={p:.3f}")

Saved figures:

- `../docs/figures/directionality_decomposition_cell16.pdf`
- `../docs/figures/wind_threshold_sensitivity_cell16.pdf`